# Task 4: Open-Set Recognition Launcher (JarvisLabs A100)
Running locally on A100.

In [ ]:
import os

# Base paths (Local to the JarvisLabs container)
TASK4_DIR  = '.'
DATA_DIR = './cifar_data'

TASK4_CKPT_DIR  = os.path.join(TASK4_DIR, 'checkpoints')
TASK4_OUT_DIR   = os.path.join(TASK4_DIR, 'results')
TASK4_CACHE_DIR = os.path.join(TASK4_DIR, 'cache')

for d in [DATA_DIR, TASK4_CKPT_DIR, TASK4_OUT_DIR, TASK4_CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Working directory: {os.path.abspath(TASK4_DIR)}')
print(f'Data will be downloaded to: {os.path.abspath(DATA_DIR)}')


### (Optional) Clean previous checkpoints and results

In [ ]:
import shutil

print('Cleaning up old Task 4 checkpoints, cache, and results...')
for d in [TASK4_CKPT_DIR, TASK4_OUT_DIR, TASK4_CACHE_DIR]:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)
print('Clean up complete!')

### Step 1a: Train Vanilla (100 epochs)

In [ ]:
!python train.py \
  --method vanilla \
  --data_root "{DATA_DIR}" \
  --checkpoints_dir "{TASK4_CKPT_DIR}" \
  --results_dir "{TASK4_OUT_DIR}"

### Step 1b: Train GCSC (100 epochs, same as Vanilla + RandAugment)

In [ ]:
!python train.py \
  --method gcsc \
  --data_root "{DATA_DIR}" \
  --checkpoints_dir "{TASK4_CKPT_DIR}" \
  --results_dir "{TASK4_OUT_DIR}"

### Step 1c: Train PROSER (50 epochs, fine-tuned from Vanilla checkpoint)

In [ ]:
!python train.py \
  --method proser \
  --data_root "{DATA_DIR}" \
  --checkpoints_dir "{TASK4_CKPT_DIR}" \
  --results_dir "{TASK4_OUT_DIR}" \
  --vanilla_ckpt "{TASK4_CKPT_DIR}/vanilla_best.pth"

### Step 2: Extract Logits and Features to Cache

In [ ]:
!python extract_outputs.py --method vanilla --data_root "{DATA_DIR}" --checkpoints_dir "{TASK4_CKPT_DIR}" --cache_dir "{TASK4_CACHE_DIR}"
!python extract_outputs.py --method gcsc --data_root "{DATA_DIR}" --checkpoints_dir "{TASK4_CKPT_DIR}" --cache_dir "{TASK4_CACHE_DIR}"
!python extract_outputs.py --method proser --data_root "{DATA_DIR}" --checkpoints_dir "{TASK4_CKPT_DIR}" --cache_dir "{TASK4_CACHE_DIR}"

### Step 3: Evaluate Open-Set Recognition

In [ ]:
!python evaluate_osr.py \
  --cache_dir "{TASK4_CACHE_DIR}" \
  --results_dir "{TASK4_OUT_DIR}" \
  --data_root "{DATA_DIR}"